In [15]:
DEVICE="mps"

In [16]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("intfloat/multilingual-e5-base", device=DEVICE)
tokenizer = encoder.tokenizer

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15128.44it/s]


In [17]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from chonkie import RecursiveChunker, SentenceChunker, TokenChunker
from IPython.display import HTML
from joblib import Memory


In [18]:
memory = Memory("cache", verbose=0)

In [19]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
def show(*, title: str, df: pd.DataFrame):
    display(HTML(f"""
        <h3 style="
            text-align: center;
            padding-bottom: 8px;
            border-bottom: 1px solid #888;
        ">{title}</h3>
    """))
    display(df)

In [20]:
docs = [json.loads(d) for d in Path("docs_clean.jsonl").read_text(encoding="utf-8").splitlines()]
queries = json.loads(Path("filmoskop_kb.json").read_text(encoding="utf-8"))["queries"]
docs_df = pd.DataFrame(docs)
print(f"Документы: {len(docs)}. Вопросы: {len(queries)}")

Документы: 173. Вопросы: 66


In [21]:
# средняя статистика по размеру текста документа в токенах
tmp = docs_df.assign(token_size=docs_df["text"].map(lambda t: len(tokenizer(t, add_special_tokens=False)["input_ids"])))
tmp["token_size"].describe()

count    173.000000
mean     182.335260
std      105.705783
min        5.000000
25%      103.000000
50%      148.000000
75%      264.000000
max      471.000000
Name: token_size, dtype: float64

Для выбора стратегии чанкинга рассмотрим следующие стратегии:
1. `Doc` без нарезки на чанки, документ == чанк.
Это основано на анализе размеров текстов документов.<br>
**max_size=471** **p75=264** говорит о том что корпус данных состоит из небольших фрагментов. И возможно, в большинстве случаев документ содержит ответ на один конкретный вопрос, а доп разбиение привидёт к потери смысла итогового вектора и ухудшит поиск. <br>Минусами такого подхода, является что он не масштабируемый, работает только для этого конкретного корпуса данных, если в корпусе будут документы размером 1000+ токенов то это сильно скажется на расходе токенов при генерации ответа, как следствие вырастет латенси. А также пострадает сам поиск, из-за размытого смысла получившегося ембеддинга по такому документу.
2. `fixed_size` - фиксированные чанки. Фиксированный размер чанка + overlap
3. `sentence` - нарезка чанков по предложениям.
4. `recursive` - рекурсивная нарезка сначала по абзацам, потом по предложениям.

В качестве реализации стратегий чанкинга я выбрал библиотеку `chonkie`



### Чанкинг

In [22]:
CHUNKERS = {
    "fixed_100": TokenChunker(tokenizer=tokenizer, chunk_size=100, chunk_overlap=20),
    "sentence_128": SentenceChunker(tokenizer=tokenizer, chunk_size=128),
    "sentence_256": SentenceChunker(tokenizer=tokenizer, chunk_size=256),
    "recursive_128": RecursiveChunker(tokenizer=tokenizer, chunk_size=128),
    "recursive_256": RecursiveChunker(tokenizer=tokenizer, chunk_size=256),

}
base = docs_df[["doc_id", "title", "category", "text"]]


def chunk_corpus(strategy: str) -> pd.DataFrame:
    if strategy == "doc":
        out = base.assign(chunk_text=base["text"])
    else:
        chunker = CHUNKERS[strategy]
        out = base.assign(chunk_text=base["text"].map(lambda t: [c.text for c in chunker(t)])).explode("chunk_text")
    out = out.drop(columns="text").assign(strategy=strategy)
    out["chunk_idx"] = out.groupby("doc_id").cumcount()
    return out

all_chunks = pd.concat([chunk_corpus(s) for s in ["doc", *CHUNKERS]], ignore_index=True)
all_chunks["token_count"] = [
    len(ids) for ids in tokenizer(all_chunks["chunk_text"].tolist(), add_special_tokens=False)["input_ids"]
]

#### Метрики чанков

In [23]:
# признаки разрыва смысла
# признак что чанк последний из всех чанков документа
is_last = all_chunks["chunk_idx"] == all_chunks.groupby(["strategy", "doc_id"])["chunk_idx"].transform("max")
# оборвано посреди предложения
all_chunks["bad_end"] = ~is_last & ~all_chunks["chunk_text"].str.contains(r"[.!?…»:]\s*$")
# начинается с середины
all_chunks["lower_start"] = (all_chunks["chunk_idx"] > 0) & all_chunks["chunk_text"].str.match(r"[а-яё]")


chunk_stats = all_chunks.groupby("strategy").agg(
    chunks=("chunk_idx", "size"),
    tokens_min=("token_count", "min"),
    tokens_median=("token_count", "median"),
    tokens_max=("token_count", "max"),
    tokens_total=("token_count", "sum"),
    short_lt30=("token_count", lambda s: (s < 30).mean()),
    bad_end=("bad_end", "mean"),
    lower_start=("lower_start", "mean"),
)
chunk_stats["dup_factor"] = chunk_stats["tokens_total"] / chunk_stats.loc["doc", "tokens_total"]
show(title="Мертрики чанков в разрезе стратегий", df=chunk_stats.round(2))

,chunks,tokens_min,tokens_median,tokens_max,tokens_total,short_lt30,bad_end,lower_start,dup_factor
strategy,,,,,,,,,
doc,173,5,148.0,471,31544,0.03,0.00,0.00,1.00
fixed_100,443,5,100.0,101,36997,0.07,0.58,0.51,1.17
recursive_128,360,5,94.0,128,31544,0.05,0.01,0.01,1.00
recursive_256,221,5,128.0,256,31544,0.06,0.00,0.00,1.00
sentence_128,336,5,105.0,128,31544,0.06,0.00,0.01,1.00
sentence_256,220,5,127.5,256,31544,0.08,0.00,0.00,1.00


In [24]:
@memory.cache
def encode(texts: list[str]) -> np.ndarray:
    """Нормализованные эмбеддинги; кэш на диске привязан к самим текстам."""
    return encoder.encode(texts, batch_size=32, normalize_embeddings=True)


In [25]:
# каждая стратегия считается без заголовка и с заголовком документа в начале чанка
variants = pd.concat([
    all_chunks.assign(with_title=False, embed_text="passage: " + all_chunks["chunk_text"]),
    all_chunks.assign(with_title=True, embed_text="passage: " + all_chunks["title"] + "\n" + all_chunks["chunk_text"]),
], ignore_index=True)

vecs = encode(variants["embed_text"].tolist())

q_df = pd.DataFrame(queries)
q_vecs = encoder.encode(("query: " + q_df["query"]).tolist(), normalize_embeddings=True)

In [26]:
# отфильтровываем только вопросы с релевантными ответами
answerable = q_df[q_df["relevant_docs"].str.len() > 0]
relevant = answerable["relevant_docs"].map(set)


def evaluate(chunks_g: pd.DataFrame, vecs_g: np.ndarray, k: int = 5) -> pd.DataFrame:
    """
    цель метрик выяснить как стратегия чанкинга влияет на результаты поиска
    логика работы:
    есть 2 стратегии создания ембеддинга:  с включением в ембеддинг поля title и без
    оптимизируем работу, сравнения всех запросов со всем набором векторов документов
    выпоняем 1 запросом
    отбор запросов по индексам dataframe работает потому что индекс 
    вектора в масиве ембеддингов запросов == индексу вопроса в dataframe
    нужно учесть что это работает если dataframe не изменяется(сортируется) после создания
    для расчёта precision используем не парметр top_k, а количество найденных уникальных документов
    """
    top = np.argsort(-(q_vecs[answerable.index] @ vecs_g.T), axis=1)[:, :k]
    found = [set(chunks_g["doc_id"].to_numpy()[row]) for row in top]
    hits = np.array([len(f & r) for f, r in zip(found, relevant)])
    tokens = chunks_g["token_count"].to_numpy()
    return pd.DataFrame({
        "recall@5": hits / relevant.map(len).to_numpy(),
        "precision@5": hits / np.array([len(f) for f in found]),
        "context_tokens@5": [tokens[row].sum() for row in top],  # сколько токенов попадёт в промпт
        "kind": answerable["kind"].to_numpy(),
        "unique_docs@5": [len(f) for f in found],
    })


per_query = pd.concat(
    {key: evaluate(g, vecs[g.index]) for key, g in variants.groupby(["strategy", "with_title"], sort=True)},
    names=["strategy", "with_title", "qi"],
).reset_index()

retrieval_table = per_query.groupby(["strategy", "with_title"], sort=False)[["recall@5", "precision@5","unique_docs@5", "context_tokens@5"]].mean().round(3)
show(title="Среднее по метрикам recall precision для всех запросов", df=retrieval_table)
show(title="Срез по типам запросов", df=per_query.pivot_table(index=["strategy", "with_title"], columns="kind", values="recall@5", sort=False).round(2))

recall@5  precision@5  unique_docs@5  \
strategy      with_title                                         
doc           False          0.688        0.175          5.000   
              True           0.670        0.175          5.000   
fixed_100     False          0.696        0.223          4.179   
              True           0.670        0.254          3.750   
recursive_128 False          0.607        0.183          4.464   
              True           0.607        0.203          4.054   
recursive_256 False          0.643        0.182          4.750   
              True           0.652        0.198          4.464   
sentence_128  False          0.634        0.187          4.482   
              True           0.643        0.210          4.125   
sentence_256  False          0.607        0.168          4.768   
              True           0.670        0.196          4.554   

                          context_tokens@5  
strategy      with_title                    
doc           False                968.589  
              True                1063.893  
fixed_100     False                414.821  
              True                 405.714  
recursive_128 False                414.089  
              True                 415.946  
recursive_256 False                746.179  
              True                 774.286  
sentence_128  False                437.982  
              True                 442.964  
sentence_256  False                738.661  
              True                 774.214

kind                      exact  semantic  multi
strategy      with_title                        
doc           False        0.90      0.62   0.59
              True         0.90      0.54   0.66
fixed_100     False        0.97      0.60   0.59
              True         0.97      0.58   0.53
recursive_128 False        0.97      0.46   0.50
              True         0.97      0.44   0.53
recursive_256 False        0.90      0.52   0.59
              True         0.97      0.48   0.62
sentence_128  False        0.97      0.50   0.53
              True         0.97      0.54   0.50
sentence_256  False        0.83      0.50   0.56
              True         0.97      0.52   0.62

## Отбор финалистов: метрики нарезки и поиска

Я сравнил шесть стратегий на `chonkie` с токенайзером энкодера `multilingual-e5-base`: `doc` (раздел целиком, эталон), `fixed_100` (окно 100 токенов, overlap 20, контроль), `sentence_128` и `sentence_256` (границы по предложениям), `recursive_128` и `recursive_256` (сначала абзацы, затем предложения). Размеры привязаны к длинам разделов в токенах: медиана 148, p75 261, максимум 471. Поэтому 128 близок к медиане, а 256 к p75.

| Стратегия | Чанков | Медиана токенов | Оборван посреди предложения | Начинается с середины | recall@5 | recall@5 с заголовком | Токенов в top-5 |
|---|---|---|---|---|---|---|---|
| doc | 173 | 148 | 0% | 0% | 0,688 | 0,670 | 969 |
| fixed_100 | 443 | 100 | 58% | 51% | 0,696 | 0,670 | 415 |
| recursive_128 | 360 | 94 | 1% | 1% | 0,607 | 0,607 | 414 |
| recursive_256 | 221 | 128 | 0% | 0% | 0,643 | 0,652 | 746 |
| sentence_128 | 336 | 105 | 0% | 1% | 0,634 | 0,643 | 438 |
| sentence_256 | 220 | 128 | 0% | 0% | 0,607 | 0,670 | 739 |

**Метрики поиска не выбирают победителя.** Я считал их на 56 запросах с ответом в базе: top-5 чанков → уникальные разделы → сравнение с `relevant_docs`. recall@5 лежит в узком диапазоне 0,61–0,70, а один запрос это до 0,018. Одна и та же нарезка `sentence_256` даёт 0,607 или 0,670 в зависимости от заголовка. Срезы по типам запросов (exact, semantic, multi на 15–25 запросах) колеблются так же. precision@5 между стратегиями я не сравниваю: у нарезок в top-5 меньше разных разделов (4,2–4,8 против 5,0 у `doc`), знаменатель уменьшается, и precision растёт без улучшения поиска.

**Заголовок в чанке.** Гипотеза: заголовок документа в начале чанка возвращает потерянное подлежащее и улучшает поиск. Эффект не подтвердился: recall меняется разнонаправленно (от −0,03 до +0,06) в пределах шума, а число разных разделов в top-5 уменьшается (у `recursive_256` с 4,75 до 4,46). Заголовок не использую.

**Финалисты: `recursive_256` и `sentence_256`.** Поскольку recall стратегии не различает, я отбирал по метрикам самой нарезки:

- у обоих нет оборванных предложений и чанков, начинающихся с середины (0%), нет overlap (раздувание текста 1,00), и меньше всего фрагментов (220–221 чанк против 336–443);
- `fixed_100` отброшен: 58% чанков оборваны, 51% начинаются с середины;
- варианты с окном 128 отброшены: больше фрагментов (336–360) и ненулевая доля разрывов (до 1%);
- `doc` не рассматриваю как кандидата: recall в пределах шума, а в промпт попадает 969 токенов против 739–746 у финалистов. Он остаётся контролем.

### Ручной просмотр

In [27]:
FINALISTS = ["sentence_256", "recursive_256"]

def handle_look(strategy: str):
    ch = all_chunks[all_chunks["strategy"] == strategy].copy()
    ch["next_text"] = ch.groupby("doc_id")["chunk_text"].shift(-1)  # следующий чанк того же документа
    multi_ids = ch.loc[ch["chunk_idx"] > 0, "doc_id"].unique()      # документы, которые реально нарезаны
    multi = ch[ch["doc_id"].isin(multi_ids)]

    # 1. конец чанка и начало следующего (10 случайных стыков)
    bounds = ch.dropna(subset=["next_text"]).sample(10, random_state=1)
    show(title=f"{strategy} - Не разорваны ли факты на границах: конец чанка и начало следующего (10 случайных стыков)", df=pd.DataFrame({
        "doc_id": bounds["doc_id"].to_numpy(),
        "конец чанка": bounds["chunk_text"].str[-100:].to_numpy(),
        "начало следующего": bounds["next_text"].str[:100].to_numpy(),
    }))

    # 2. Не потерялось ли подлежащее: чанки, которые начинаются с местоимения или с маленькой буквы
    PRONOUN = r"^(он|она|оно|они|это|этот|эта|эти|такой|также|поэтому|там|тогда)\b"
    starts = multi[multi["chunk_idx"] > 0]
    text = starts["chunk_text"]
    bad_start = starts[text.str.lower().str.match(PRONOUN) | text.str.match(r"[а-яё]")]
    show(title=f"{strategy} - Без подлежащего в начале: {len(bad_start)} из {len(starts)} чанков", df=bad_start[["doc_id", "title", "chunk_text"]])

    # 3. Сохраняется ли смысл без соседей: самые короткие чанки и несколько документов целиком
    show(title=f"{strategy} - Сохраняется ли смысл без соседей: самые короткие чанки и несколько документов целиком", df=multi.nsmallest(8, "token_count")[["doc_id", "title", "token_count", "chunk_text"]])

    for doc_id in multi["doc_id"].drop_duplicates().sample(3, random_state=1):
        display(multi[multi["doc_id"] == doc_id][["doc_id", "title", "chunk_idx", "chunk_text"]])

for strategy in FINALISTS:
    handle_look(strategy)

,doc_id,конец чанка,начало следующего
0,91,"дакция собирает на главной — загляните в её полки, там есть и «Уютный вечер», и подборки поспорить.","А если собраться в одной комнате не выходит, посмотрите раздел справки «Совместный просмотр»."
1,81,"равится» тоже стоит ставить не из вежливости: они меняют то, что сервис предложит в следующий раз.\n\n","Если название известно заранее, искать его быстрее напрямую — про это раздел «Поиск по каталогу»."
2,161,"подписан во время просмотра, эта настройка не влияет: интерфейс и контент локализуются независимо.\n\n","Насколько подробно локализована карточка, зависит от того, что передал правообладатель: у части ката"
3,144,- у части контента нет прав на офлайн-просмотр\n- на одном устройстве может храниться до 20 загрузок\n,- скачанный файл доступен до 30 дней\n- при подключении к интернету срок продлевается автоматически\n\n
4,5,"описаны в разделе об отмене подписки, и до их окончания обращение всё равно останется в ожидании.\n\n","Где посмотреть сведения о прошлых списаниях в самом сервисе, описано в отдельном разделе справки об"
5,20,"\n\nСтрана определяется по подключению: сервис ориентируется на сеть, из которой вы вышли в интернет.","Поэтому каталог меняется после подключения к местному Wi-Fi или мобильной сети, а не в момент пересе"
6,110,"сли хотя бы одно звено не поддерживает требования, сервис автоматически снизит качество до Full HD.","Фильмы с поддержкой 4K отмечены в карточке значком «4K».\nПроверить, что именно воспроизводится сейча"
7,128,ехода виден в разделе «Подписка»: там появляются название нового тарифа и дата следующего списания.,"Если настройки показывают прежний тариф, обновите страницу или перезапустите приложение — данные под"
8,173,"список устройств и отвяжите незнакомые. 4. Проверьте, не изменились ли привязанные почта и телефон.\n","Если изменить пароль не удаётся — доступ уже потерян, — обратитесь в поддержку через форму на сайте."
9,104,"а живёт своей жизнью: все настройки, списания и уведомления остаются на стороне владельца аккаунта.",Управлять этой подпиской из своего аккаунта не получится.


,doc_id,title,chunk_text
976,20,Просмотр за границей,"Поэтому каталог меняется после подключения к местному Wi-Fi или мобильной сети, а не в момент пересечения границы.\n\nЕсли нужного фильма в каталоге страны пребывания нет, добавьте его в список «Моё» — карточка сохранится, и после возвращения фильм снова станет доступен. Заново искать его в каталоге не придётся. То же касается сериалов: отмеченные позиции никуда не пропадают из списка.\n\nКогда после возвращения домой каталог остался прежним, перезапустите приложение и проверьте, что подключение идёт через домашнюю сеть. Если это не помогает, напишите в поддержку и укажите страну, из которой заходите."
1095,116,Часто задаваемые вопросы об оплате,"Это особенность отображения в банковском приложении, на сами операции она не влияет."
1170,179,Как выбрать HDMI-кабель,"Такой кабель отработал своё: его дешевле заменить, чем искать причину в настройках.\n\nСтоит посмотреть и на сами разъёмы: у телевизора их несколько, и они не всегда равнозначны. Какой порт основной, написано в инструкции."


,doc_id,title,token_count,chunk_text
1155,168,Первые дни в «Фильмоскопе»,5,Приятного просмотра.
1080,104,Можно ли подарить подписку,12,Управлять этой подпиской из своего аккаунта не получится.
1140,157,Что сервис знает о ваших просмотрах,19,"Как их посмотреть и что с ними можно сделать, описано в отдельном разделе справки."
1055,81,"Как выбрать, что посмотреть",20,"Если название известно заранее, искать его быстрее напрямую — про это раздел «Поиск по каталогу»."
1095,116,Часто задаваемые вопросы об оплате,20,"Это особенность отображения в банковском приложении, на сами операции она не влияет."
996,35,Почему банк отклоняет платёж,23,"Порядок действий на стороне сервиса, когда списание не удалось, описан в отдельном разделе справки об оплате."
1066,91,Смотрим вместе: идеи для киновечера,23,"А если собраться в одной комнате не выходит, посмотрите раздел справки «Совместный просмотр»."
1022,56,4K и HDR: что это такое,25,"Что должно совпасть, чтобы фильм действительно пошёл в 4K HDR, — тема отдельного раздела справки."


,doc_id,title,chunk_idx,chunk_text
1065,91,Смотрим вместе: идеи для киновечера,0,"Киновечер — это не «включить что-нибудь фоном», а маленькое событие, ради которого люди договариваются о времени.\n\nНачните с темы. Один вечер — одно настроение: нуар под дождь за окном, ретро-комедии, мюзиклы, «фильмы, которые все смотрели, а мы почему-то нет». Тема снимает главную беду совместного просмотра — получасовые переговоры у главного экрана, после которых смотреть уже поздно.\n\nДоговоритесь, кто выбирает. Очередь по кругу работает лучше голосования: сегодня выбираете вы, в следующий раз — гость, который весь вечер ворчал. Право на один вечер и право на одну ошибку.\n\nНе забудьте про антураж: приглушённый свет, плед, попкорн и нарезка на общей тарелке, телефоны экраном вниз. Титры — тоже часть вечера, под них хорошо начинается разговор о фильме.\n\nГотовые темы для общей компании редакция собирает на главной — загляните в её полки, там есть и «Уютный вечер», и подборки поспорить."
1066,91,Смотрим вместе: идеи для киновечера,1,"А если собраться в одной комнате не выходит, посмотрите раздел справки «Совместный просмотр»."


,doc_id,title,chunk_idx,chunk_text
1054,81,"Как выбрать, что посмотреть",0,"Каталог большой, и выбор фильма на вечер иногда занимает больше времени, чем сам вечер. Несколько способов сузить его.\n\nПолки на главном экране. Главная собрана из тематических полок: новинки, популярное у зрителей, подборки редакции и персональные рекомендации профиля. Полки редакции одинаковы для всех, персональные — свои у каждого профиля.\n\nЖанры. В каталоге фильмы и сериалы разложены по жанрам, а внутри жанра список сортируется по году выхода или по популярности. Способ рабочий, когда настроение понятно, а название — нет: «хочу комедию, но не свежую».\n\nКарточка фильма. В ней собрано то, по чему обычно и принимают решение: возрастной рейтинг, год, страна, продолжительность, жанры и короткое описание без спойлеров.\n\nСписок «Моё». Всё, что заинтересовало, но сегодня не подошло, складывайте туда — через неделю у вас будет собственная полка на вечер. Оценки «Нравится» и «Не нравится» тоже стоит ставить не из вежливости: они меняют то, что сервис предложит в следующий раз.\n\n"
1055,81,"Как выбрать, что посмотреть",1,"Если название известно заранее, искать его быстрее напрямую — про это раздел «Поиск по каталогу»."


,doc_id,title,chunk_idx,chunk_text
1144,161,Языки и локализация каталога,0,"Каталог «Фильмоскопа» собран для русскоязычного зрителя, поэтому почти всё, что вы видите на экране, переведено.\n\nНазвания. Зарубежные фильмы и сериалы подписаны прокатным русским названием — тем, под которым картина выходила в России. Оригинальное название указано в карточке рядом с годом выпуска: по нему проще найти знакомую картину, если прокатное название с ней расходится. Поиск понимает оба варианта, а часто и написание латиницей.\n\nОписания и материалы. Синопсис, жанры, имена актёров и режиссёров редакция ведёт на русском. Постеры и кадры берутся из локализованных материалов правообладателя, если они есть, — иначе показываются оригинальные.\n\nЯзык интерфейса. Он задаётся в настройках профиля и отвечает только за подписи меню, кнопок и разделов. На состав каталога и на то, как фильм звучит и подписан во время просмотра, эта настройка не влияет: интерфейс и контент локализуются независимо.\n\n"
1145,161,Языки и локализация каталога,1,"Насколько подробно локализована карточка, зависит от того, что передал правообладатель: у части каталога описания короткие, а постеры остаются оригинальными."


... (2 outputs left out)

,doc_id,title,token_count,chunk_text
1736,168,Первые дни в «Фильмоскопе»,5,Приятного просмотра.
1635,81,"Как выбрать, что посмотреть",20,"Если название известно заранее, искать его быстрее напрямую — про это раздел «Поиск по каталогу»."
1576,35,Почему банк отклоняет платёж,23,"Порядок действий на стороне сервиса, когда списание не удалось, описан в отдельном разделе справки об оплате."
1540,5,Финансовые вопросы: куда обратиться,26,"Где посмотреть сведения о прошлых списаниях в самом сервисе, описано в отдельном разделе справки об оплате."
1585,42,Один аккаунт на всю семью,26,"Если аккаунтов в семье почему-то оказалось несколько, что с ними делать — разобрано в отдельном разделе справки."
1624,72,Почему не удаётся войти в аккаунт,28,"Если пароль не подходит ни в одном варианте, дело уже не в опечатке — порядок восстановления описан в отдельном разделе справки."
1616,66,Приложения нет в магазине телевизора,29,"Поддерживается ли ваша модель, написано в разделе справки со списком устройств — начинать проще с него, а не с настроек."
1721,157,Что сервис знает о ваших просмотрах,29,"От зрителя эти записи не спрятаны. Как их посмотреть и что с ними можно сделать, описано в отдельном разделе справки."


,doc_id,title,chunk_idx,chunk_text
1645,91,Смотрим вместе: идеи для киновечера,0,"Киновечер — это не «включить что-нибудь фоном», а маленькое событие, ради которого люди договариваются о времени.\n\nНачните с темы. Один вечер — одно настроение: нуар под дождь за окном, ретро-комедии, мюзиклы, «фильмы, которые все смотрели, а мы почему-то нет». Тема снимает главную беду совместного просмотра — получасовые переговоры у главного экрана, после которых смотреть уже поздно.\n\nДоговоритесь, кто выбирает. Очередь по кругу работает лучше голосования: сегодня выбираете вы, в следующий раз — гость, который весь вечер ворчал. Право на один вечер и право на одну ошибку.\n\nНе забудьте про антураж: приглушённый свет, плед, попкорн и нарезка на общей тарелке, телефоны экраном вниз. Титры — тоже часть вечера, под них хорошо начинается разговор о фильме.\n\n"
1646,91,Смотрим вместе: идеи для киновечера,1,"Готовые темы для общей компании редакция собирает на главной — загляните в её полки, там есть и «Уютный вечер», и подборки поспорить. А если собраться в одной комнате не выходит, посмотрите раздел справки «Совместный просмотр»."


,doc_id,title,chunk_idx,chunk_text
1634,81,"Как выбрать, что посмотреть",0,"Каталог большой, и выбор фильма на вечер иногда занимает больше времени, чем сам вечер. Несколько способов сузить его.\n\nПолки на главном экране. Главная собрана из тематических полок: новинки, популярное у зрителей, подборки редакции и персональные рекомендации профиля. Полки редакции одинаковы для всех, персональные — свои у каждого профиля.\n\nЖанры. В каталоге фильмы и сериалы разложены по жанрам, а внутри жанра список сортируется по году выхода или по популярности. Способ рабочий, когда настроение понятно, а название — нет: «хочу комедию, но не свежую».\n\nКарточка фильма. В ней собрано то, по чему обычно и принимают решение: возрастной рейтинг, год, страна, продолжительность, жанры и короткое описание без спойлеров.\n\nСписок «Моё». Всё, что заинтересовало, но сегодня не подошло, складывайте туда — через неделю у вас будет собственная полка на вечер. Оценки «Нравится» и «Не нравится» тоже стоит ставить не из вежливости: они меняют то, что сервис предложит в следующий раз.\n\n"
1635,81,"Как выбрать, что посмотреть",1,"Если название известно заранее, искать его быстрее напрямую — про это раздел «Поиск по каталогу»."


,doc_id,title,chunk_idx,chunk_text
1725,161,Языки и локализация каталога,0,"Каталог «Фильмоскопа» собран для русскоязычного зрителя, поэтому почти всё, что вы видите на экране, переведено.\n\nНазвания. Зарубежные фильмы и сериалы подписаны прокатным русским названием — тем, под которым картина выходила в России. Оригинальное название указано в карточке рядом с годом выпуска: по нему проще найти знакомую картину, если прокатное название с ней расходится. Поиск понимает оба варианта, а часто и написание латиницей.\n\nОписания и материалы. Синопсис, жанры, имена актёров и режиссёров редакция ведёт на русском. Постеры и кадры берутся из локализованных материалов правообладателя, если они есть, — иначе показываются оригинальные.\n\nЯзык интерфейса. Он задаётся в настройках профиля и отвечает только за подписи меню, кнопок и разделов. На состав каталога и на то, как фильм звучит и подписан во время просмотра, эта настройка не влияет: интерфейс и контент локализуются независимо.\n\n"
1726,161,Языки и локализация каталога,1,"Насколько подробно локализована карточка, зависит от того, что передал правообладатель: у части каталога описания короткие, а постеры остаются оригинальными."


## Выбор победителя: ручной просмотр финалистов

Для каждого финалиста я проверил три пункта: 10 случайных стыков соседних чанков (не разорваны ли факты на границах), чанки, начинающиеся с местоимения или с маленькой буквы (не потерялось ли подлежащее), 8 самых коротких чанков и три нарезанных документа целиком (сохраняется ли смысл без соседей). Выборки у обеих нарезок одинаковые (`random_state=1`).

| | `recursive_256` | `sentence_256` |
|---|---|---|
| Стыки на границе абзаца или строки | 9 из 10 | 4 из 10 |
| Стыки внутри абзаца или списка | 1 (список в doc 144) | 6 (docs 91, 20, 110, 128, 104 и список в doc 144) |
| Чанки без подлежащего в начале | 0 из 48 | 3 из 47 (6%) |
| Самые короткие чанки | 5–29 токенов, абзацы-ссылки | 5–25 токенов, в том числе оторванные концы абзацев |

**Границы.** `recursive_256` режет по границам абзацев. `sentence_256` режет и внутри абзаца, разделяя причину и следствие: в doc 20 фраза «Страна определяется по подключению…» оторвана от «Поэтому каталог меняется после подключения к местному Wi-Fi…», в doc 128 фраза «…дата следующего списания» оторвана от «Если настройки показывают прежний тариф, обновите страницу…».

**Подлежащее.** У `recursive_256` таких чанков нет. У `sentence_256` три начинаются без подлежащего: «Поэтому каталог меняется…» (doc 20), «Это особенность отображения в банковском приложении…» (doc 116), «Такой кабель отработал своё…» (doc 179).

**Смысл без соседей.** В обеих нарезках короткие чанки это абзацы-ссылки вроде «…про это раздел «Поиск по каталогу»» (doc 81) или «Приятного просмотра.» (doc 168), без соседей они ничего не отвечают. У `sentence_256` к ним добавляются оторванные концы абзацев: в doc 104 «Управлять этой подпиской из своего аккаунта не получится.» (12 токенов) отделена от абзаца про оплату в чужом аккаунте, в doc 91 оторвана последняя фраза абзаца («А если собраться в одной комнате не выходит, посмотрите раздел справки…»). В `recursive_256` те же абзацы остаются целыми. Из трёх нарезанных документов, просмотренных целиком, два (81 и 161) нарезаны одинаково, а в третьем (91) `sentence_256` хуже.

**Общее ограничение.** В обеих нарезках список в doc 144 разрезан между пунктами «до 20 загрузок» и «доступен до 30 дней».

**Победитель: `recursive_256`.** По метрикам поиска финалисты неразличимы (recall 0,643 против 0,607, это около двух запросов из 56, контекст 746 и 739 токенов). Разницу создаёт только ручной просмотр: `recursive_256` не рвёт абзацы и не теряет подлежащее.

### Сохранение чанков и эмбеддингов

In [28]:
CHOSEN = "recursive_256"
MODEL_NAME = "intfloat/multilingual-e5-base"

# 1. Чанки победителя с метаданными и полным текстом
final = (
    all_chunks[all_chunks["strategy"] == CHOSEN]
    [["doc_id", "title", "category", "chunk_idx", "chunk_text", "token_count"]]
    .reset_index(drop=True)
)
final.insert(0, "chunk_id", final["doc_id"].astype(str) + ":" + final["chunk_idx"].astype(str))
final.to_json("chunks.jsonl", orient="records", lines=True, force_ascii=False)

# 2. Эмбеддинги считаем по сохранённому файлу
chunks_saved = pd.read_json("chunks.jsonl", lines=True)
chunk_vecs = encode(("passage: " + chunks_saved["chunk_text"]).tolist())
np.save("chunk_embeddings.npy", chunk_vecs)

# 3. Проверки: файл не исказил текст, строка i = вектор i, вектора те же, на которых считались метрики
assert chunks_saved["chunk_text"].tolist() == final["chunk_text"].tolist()
assert chunk_vecs.shape[0] == len(chunks_saved)
ref = vecs[variants.index[(variants["strategy"] == CHOSEN) & ~variants["with_title"]]]
assert np.allclose(chunk_vecs, ref, atol=1e-4)

# 4. Параметры, чтобы этап 3 строил запросы так же
Path("chunking_meta.json").write_text(json.dumps({
    "strategy": CHOSEN,
    "chunker": "chonkie.RecursiveChunker",
    "chunk_size_tokens": 256,
    "overlap": 0,
    "with_title": False,
    "embedding_model": MODEL_NAME,
    "passage_prefix": "passage: ",
    "query_prefix": "query: ",
    "normalized": True,
    "chunks": len(chunks_saved),
    "dim": int(chunk_vecs.shape[1]),
}, ensure_ascii=False, indent=2), encoding="utf-8")

show(title="Сохранённые чанки (первые 3)", df=chunks_saved.head(3))
show(title="Размеры", df=pd.Series({"chunks": len(chunks_saved), "embedding_shape": str(chunk_vecs.shape)}).to_frame("value"))

,chunk_id,doc_id,title,category,chunk_idx,chunk_text,token_count
0,0:0,0,Промокоды,Подписка и оплата,0,"Промокод даёт скидку на подписку или бесплатные дни. Ввести его можно в разделе «Подписка» — поле «У меня есть промокод» — или при оформлении подписки.\n\nПравила простые: один промокод на один аккаунт, скидки по промокодам не суммируются между собой и с другими акциями, у каждого кода есть срок действия — он указан в условиях акции. Если код не применяется, проверьте срок действия и то, что он ещё не был использован на этом аккаунте.",113
1,1:0,1,Синхронизация между устройствами,Функции и контент,0,"Всё, что вы делаете в «Фильмоскопе», хранится не на устройстве, а в аккаунте. Поэтому телефон, планшет, браузер и телевизор показывают одно и то же: вошли — и сервис выглядит привычно, даже если устройство чужое.\n\nОбщее для аккаунта: подписка и тариф, способ оплаты, список привязанных устройств. Это не зависит ни от профиля, ни от того, откуда вы зашли.\n\nОбщее внутри профиля: список «Моё», оценки «Нравится» и «Не нравится», купленные фильмы и настройки самого профиля — язык интерфейса, автовоспроизведение, аватар. Отметили фильм на телефоне — он окажется в «Моём» и на телевизоре. При этом у каждого профиля всё своё: соседний профиль ваших оценок и вашего списка не увидит.\n\nОстаётся на устройстве: скачанные файлы и настройки, которые относятся к самой технике, — громкость, вывод звука, параметры изображения. Переносить их некуда, поэтому на новом устройстве они задаются заново.\n\n",236
2,1:1,1,Синхронизация между устройствами,Функции и контент,1,"Синхронизация требует интернета. Если устройство было офлайн, изменения доедут до аккаунта при следующем подключении, а пока экраны на разных устройствах могут расходиться.",40


,value
chunks,221
embedding_shape,"(221, 768)"


## Итоговый отчёт

**Что сделал.** Нарезал очищенный корпус (173 раздела) шестью стратегиями, построил эмбеддинги `multilingual-e5-base` без заголовка и с заголовком документа, посчитал метрики нарезки и поиска, отобрал двух финалистов и выбрал победителя ручным просмотром.

**Как выбирал.**

1. Метрики поиска (recall@5 0,61–0,70) стратегии не различают: разница в один-два запроса из 56 это шум. Заголовок в чанке на поиск систематически не влияет.
2. По метрикам нарезки отсеялись `fixed_100` (58% оборванных чанков), варианты с окном 128 (больше фрагментов) и `doc` (969 токенов контекста без выигрыша в поиске). Финалисты: `recursive_256` и `sentence_256`.
3. Ручной просмотр границ, подлежащего и смысла без соседей показал, что `recursive_256` лучше сохраняет целостность чанков: 0 чанков без подлежащего против 3 из 47 и 9 из 10 чистых стыков против 4 из 10.

Расхождения между стратегиями не глобальные: только по цифрам победителя выбрать сложно. Весомее сочетание цифр (целостность границ, число фрагментов, цена контекста) и разбора конкретных чанков, на него я и опираюсь.

**Выбор: `recursive_256` без заголовка.** `RecursiveChunker` из `chonkie`, окно 256 токенов, без overlap: режет по абзацам, затем по строкам и предложениям. 221 чанк на 173 раздела, медиана 128 токенов, максимум 256. Весь текст сохранён (31 544 токена, как в корпусе), оборванных предложений и чанков без подлежащего нет, в top-5 попадает в среднем 746 токенов.

**Ограничения.**

- 56 запросов выявляют только крупные эффекты.
- Список в doc 144 разрезан между чанками, а короткие чанки-ссылки (до 29 токенов) без контекста ничего не отвечают.
- recall считается по разделам и не видит оборванных фактов, поэтому качество чанков для генерации проверю на этапах 5–6.
- Размер окна выбран по длинам разделов, а не перебором, на другом корпусе его надо пересматривать.
- Короткие заглушки и архивные записи попадают в топ при любой стратегии, возвращаюсь к ним на этапе 4.

**Результат этапа.** `chunks.jsonl` (221 чанк: `chunk_id`, `doc_id`, `title`, `category`, `chunk_idx`, `chunk_text`, `token_count`), `chunk_embeddings.npy` (221 × 768, нормализованные, префикс `passage: `) и `chunking_meta.json` (параметры нарезки и модели).